 # MEAN across all stations (Group by date + County )

In [ ]:
import pandas as pd

# 1. Load the original EPA CO file
pm25 = pd.read_csv("908523 Swarnali Mollick SIL Submission/Raw CSVs/AQI_2020/NY_PM2_5_2020.csv")   # change path if needed

# 2. Create a proper datetime column from 'Date'
pm25["date"] = pd.to_datetime(pm25["Date"], format="%m/%d/%Y")

# 3. Group by date + County and take the MEAN across all stations
#    for BOTH concentration and Daily AQI Value
pm25_daily = (
    pm25
    .groupby(["date", "County"], as_index=False)
    .agg({
        "Daily Mean PM2.5 Concentration": "mean",
        "Daily AQI Value": "max"
    })
)

# 4. Rename columns to cleaner names
pm25_daily = pm25_daily.rename(columns={
    "County": "city",
    "Daily Mean PM2.5 Concentration": "pm25_mean",
    "Daily AQI Value": "pm25_aqi"
})

# 5. Clip tiny negative values (if any) to 0
pm25_daily["pm25_mean"] = pm25_daily["pm25_mean"].clip(lower=0)

# 6. Round mean columns to 2 decimal places
pm25_daily["pm25_mean"] = pm25_daily["pm25_mean"].round(2)
pm25_daily["pm25_aqi"]  = pm25_daily["pm25_aqi"].round(2)

# 7. Sort by city (A→Z) and then by date (oldest→newest)
pm25_daily = pm25_daily.sort_values(
    by=["city", "date"],
    ascending=[True, True]
).reset_index(drop=True)

# 8. Save the result as a clean city–day file
pm25_daily.to_csv("908523 Swarnali Mollick SIL Submission/Raw CSVs/CleanData_city_aqi/NY_PM2_5_daily_2020.csv", index=False)

print(pm25_daily.head())


        date    city  pm25_mean  pm25_aqi
0 2020-01-01  Albany       2.75        22
1 2020-01-02  Albany       8.15        50
2 2020-01-03  Albany      12.35        59
3 2020-01-04  Albany      10.06        58
4 2020-01-05  Albany       2.50        16


# MERGE POLLUTANTS (After Calculating Mean) + WEATHER

In [ ]:
import pandas as pd
import os

target_cities = ["Bronx", "Erie", "Monroe", "Queens"]

# ---------------------------
# 2. LOAD POLLUTANT TABLES (city-daily)
# ---------------------------
root = r"908523 Swarnali Mollick SIL Submission/Raw CSVs/CleanData_city_aqi"
co   = pd.read_csv(os.path.join(root,"NY_CO_daily_2020.csv"),   parse_dates=["date"])
pm25 = pd.read_csv(os.path.join(root,"NY_PM2_5_daily_2020.csv"), parse_dates=["date"])
o3   = pd.read_csv(os.path.join(root,"NY_O3_daily_2020.csv"),   parse_dates=["date"])
no2  = pd.read_csv(os.path.join(root,"NY_NO2_daily_2020.csv"),  parse_dates=["date"])
so2  = pd.read_csv(os.path.join(root,"NY_SO2_daily_2020.csv"),  parse_dates=["date"])  # or NY_so2_... if that's your name

# keep only Queens & Bronx in each
co   = co[co["city"].isin(target_cities)].copy()
pm25 = pm25[pm25["city"].isin(target_cities)].copy()
o3   = o3[o3["city"].isin(target_cities)].copy()
no2  = no2[no2["city"].isin(target_cities)].copy()
so2  = so2[so2["city"].isin(target_cities)].copy()

# ---------------------------
# 3. MERGE POLLUTANTS INTO ONE TABLE
# ---------------------------
# Start from CO and merge others on (date, city)
pollutants = co.merge(
    no2[["date", "city", "no2_mean", "no2_aqi"]],
    on=["date", "city"],
    how="outer"
)

pollutants = pollutants.merge(
    so2[["date", "city", "so2_mean", "so2_aqi"]],
    on=["date", "city"],
    how="outer"
)

pollutants = pollutants.merge(
    pm25[["date", "city", "pm2_5_mean", "pm2_5_aqi"]],
    on=["date", "city"],
    how="outer"
)

pollutants = pollutants.merge(
    o3[["date", "city", "o3_mean", "o3_aqi"]],
    on=["date", "city"],
    how="outer"
)

# Optional: sort for readability
pollutants = pollutants.sort_values(["city", "date"]).reset_index(drop=True)

print("Pollutant table shape:", pollutants.shape)
print(pollutants.head())

# ---------------------------
# 4. PREPARE WEATHER DATA (Queens + Bronx + "Erie" + "Monroe")
# ---------------------------
def prepare_weather(path, city_name):
    """Load Visual Crossing daily weather and keep key columns."""
    w = pd.read_csv(path)
    # adjust the column name if your date column is different
    w["date"] = pd.to_datetime(w["datetime"])
    w["city"] = city_name
    
    # keep the weather features you care about
    # check your CSV to confirm these column names: temp, humidity, windspeed
    cols = ["date", "city", "temp", "humidity", "windspeed"]
    w_small = w[cols].copy()
    return w_small

root = r"908523 Swarnali Mollick SIL Submission/Raw CSVs/W_2020/"
w_bronx   = prepare_weather(os.path.join(root, "Weather_Bronx2020.csv"),   "Bronx")
w_erie  = prepare_weather(os.path.join(root, "Weather_Erie2020.csv"),  "Erie")
w_monroe = prepare_weather(os.path.join(root, "Weather_Monroe2020.csv"), "Monroe")
w_queens  = prepare_weather(os.path.join(root, "Weather_Queens2020.csv"),  "Queens")

weather = pd.concat([w_bronx, w_erie, w_monroe, w_queens], ignore_index=True)

print("Weather table shape:", weather.shape)
print(weather.head())

# ---------------------------
# 5. MERGE POLLUTANTS + WEATHER
# ---------------------------
df_qb = pollutants.merge(
    weather,
    on=["date", "city"],
    how="inner"   # keep days where BOTH pollution + weather exist
)

# Sort nicely
df_qb = df_qb.sort_values(["city", "date"]).reset_index(drop=True)

print("Final merged table (Bronx + Erie + Monroe + Queens) shape:", df_qb.shape)
print(df_qb.head())

# ---------------------------
# 6. SAVE FINAL DATASET
# ---------------------------
df_qb.to_csv("908523 Swarnali Mollick SIL Submission/Raw CSVs/NY_BEMQ_pollution_weather_2020.csv", index=False)


Pollutant table shape: (1464, 12)
        date   city  co_mean  co_aqi  no2_mean  no2_aqi  so2_mean  so2_aqi  \
0 2020-01-01  Bronx      0.2     2.0     16.25     17.0      0.55      0.0   
1 2020-01-02  Bronx      0.5     6.0     39.95     38.0      2.70      4.0   
2 2020-01-03  Bronx      0.6     7.0     36.95     36.0      1.40      1.0   
3 2020-01-04  Bronx      0.7     8.0     27.90     28.0      1.20      1.0   
4 2020-01-05  Bronx      0.3     3.0     20.35     19.0      0.60      0.0   

   pm2_5_mean  pm2_5_aqi  o3_mean  o3_aqi  
0        4.68         42     0.02    19.0  
1       11.85         59      NaN     NaN  
2       16.23         68     0.00     1.0  
3       11.93         58     0.02    22.0  
4        4.43         37     0.03    26.0  
Weather table shape: (1464, 5)
        date   city  temp  humidity  windspeed
0 2020-01-01  Bronx  38.2      50.3       18.6
1 2020-01-02  Bronx  41.3      52.2       10.7
2 2020-01-03  Bronx  46.8      79.2        6.5
3 2020-01-04  

# Annotation Dataset

In [ ]:
import pandas as pd

# =========================
# 1) Load data
# =========================
df = pd.read_csv("908523 Swarnali Mollick SIL Submission/Raw CSVs/NY_BEMQ_pollution_weather_2020.csv")

# Clean column names (fix hidden spaces / weird chars)
df.columns = df.columns.str.strip()

# Date
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# =========================
# 2) AQI columns
# =========================
aqi_cols = ["co_aqi", "no2_aqi", "so2_aqi", "pm2_5_aqi", "o3_aqi"]

# Check columns exist (if not, you'll see exactly which)
missing = [c for c in aqi_cols if c not in df.columns]
if missing:
    raise ValueError(
        f"Missing AQI columns in CSV: {missing}\n"
        f"Available columns are:\n{df.columns.tolist()}"
    )

df[aqi_cols] = df[aqi_cols].replace(
    to_replace=[r"^\s*$", "NA", "N/A", "na", "n/a", "--", "-", "None", "null"],
    value=pd.NA,
    regex=True
)

df[aqi_cols] = df[aqi_cols].apply(pd.to_numeric, errors="coerce")

# =========================
# 4) Compute max AQI per row
# =========================
df["max_aqi"] = df[aqi_cols].max(axis=1, skipna=True)

all_missing_rate = df[aqi_cols].isna().all(axis=1).mean()
print(f"Rows where ALL 5 AQI values are missing: {all_missing_rate:.2%}")

# =========================
# 5) Binary label based on max_aqi
# =========================
def annotate_aqi(aqi):
    if pd.isna(aqi):
        return pd.NA  # keep missing if max_aqi missing
    return 0 if 0 <= aqi <= 50 else 1

df["aqi_level"] = df["max_aqi"].apply(annotate_aqi)

aqi_map = {0: "healthy", 1: "unhealthy"}
df["aqi_level_str"] = df["aqi_level"].map(aqi_map)

# =========================
# 6) Main pollutant (argmax of AQI columns)
# =========================
pollutant_map = {
    "co_aqi": "CO",
    "no2_aqi": "NO2",
    "so2_aqi": "SO2",
    "pm2_5_aqi": "PM2.5",
    "o3_aqi": "O3"
}

# idxmax returns a column name; if all are NaN in a row, idxmax can misbehave.
# So we mask those rows first.
all_nan_mask = df[aqi_cols].isna().all(axis=1)
df.loc[~all_nan_mask, "Main_Pollutant"] = (
    df.loc[~all_nan_mask, aqi_cols]
      .idxmax(axis=1)
      .map(pollutant_map)
)
df.loc[all_nan_mask, "Main_Pollutant"] = pd.NA

# =========================
# 7) Preview
# =========================
print(df[["date", "city", "max_aqi", "aqi_level_str", "Main_Pollutant"]].head(10))

# =========================
# 8) Save
# =========================
df.to_csv("908523 Swarnali Mollick SIL Submission/Raw CSVs/NY_BEMQ_pollution_weather_2020_annotated.csv", index=False)
print("Saved: NY_BEMQ_pollution_weather_2020_annotated.csv")


Rows where ALL 5 AQI values are missing: 0.00%
        date   city  max_aqi aqi_level_str Main_Pollutant
0 2020-01-01  Bronx     42.0       healthy          PM2.5
1 2020-01-02  Bronx     59.0     unhealthy          PM2.5
2 2020-01-03  Bronx     68.0     unhealthy          PM2.5
3 2020-01-04  Bronx     58.0     unhealthy          PM2.5
4 2020-01-05  Bronx     37.0       healthy          PM2.5
5 2020-01-06  Bronx     54.0     unhealthy          PM2.5
6 2020-01-07  Bronx     51.0     unhealthy          PM2.5
7 2020-01-08  Bronx     48.0       healthy          PM2.5
8 2020-01-09  Bronx     44.0       healthy          PM2.5
9 2020-01-10  Bronx     55.0     unhealthy          PM2.5
Saved: NY_BEMQ_pollution_weather_2020_annotated.csv


# Merging Years

In [ ]:
import pandas as pd

# =========================
# 1) Read both years
# =========================
df_2020 = pd.read_csv("908523 Swarnali Mollick SIL Submission/Raw CSVs/NY_BEMQ_pollution_weather_2020_annotated.csv")
df_2024 = pd.read_csv("908523 Swarnali Mollick SIL Submission/Raw CSVs/NY_BEMQ_pollution_weather_2024_annotated.csv")

# Clean column names (handles hidden spaces)
df_2020.columns = df_2020.columns.str.strip()
df_2024.columns = df_2024.columns.str.strip()

# Parse date (optional but recommended)
if "date" in df_2020.columns:
    df_2020["date"] = pd.to_datetime(df_2020["date"], errors="coerce")
if "date" in df_2024.columns:
    df_2024["date"] = pd.to_datetime(df_2024["date"], errors="coerce")

# Add year column (useful for analysis)
df_2020["year"] = 2020
df_2024["year"] = 2024

base_cols = list(df_2024.columns)  # keep the same order as 2024 file

# Add any columns that exist in 2020 but not in 2024 (append at end for now)
extra_2020_cols = [c for c in df_2020.columns if c not in base_cols]
final_cols = base_cols + extra_2020_cols

# Ensure both dataframes have all columns (missing -> NA)
for c in final_cols:
    if c not in df_2020.columns:
        df_2020[c] = pd.NA
    if c not in df_2024.columns:
        df_2024[c] = pd.NA

# Reindex both to the same column order
df_2020 = df_2020[final_cols]
df_2024 = df_2024[final_cols]

# =========================
# 3) Merge (stack rows)
# =========================
df_all = pd.concat([df_2020, df_2024], ignore_index=True)


tail_cols = ["Main_Pollutant", "aqi_level", "aqi_level_str"]

# Keep only those that actually exist (prevents KeyError)
tail_cols_exist = [c for c in tail_cols if c in df_all.columns]
other_cols = [c for c in df_all.columns if c not in tail_cols_exist]

df_all = df_all[other_cols + tail_cols_exist]

# =========================
# 5) Save
# =========================
out_file = "908523 Swarnali Mollick SIL Submission/rmd_final/NY_BEMQ_pollution_weather_2020_2024_annotated_merged.csv"
df_all.to_csv(out_file, index=False)

print("Saved:", out_file)
print("Final columns (last 10):", df_all.columns.tolist()[-10:])
print("Shape:", df_all.shape)


Saved: C:/Users/SwarnaliMollick/Downloads/SIL Project/NY_BEMQ_pollution_weather_2020_2024_annotated_merged.csv
Final columns (last 10): ['o3_mean', 'o3_aqi', 'temp', 'humidity', 'windspeed', 'max_aqi', 'year', 'Main_Pollutant', 'aqi_level', 'aqi_level_str']
Shape: (2928, 20)
